# Enseñame la Pasta

### 1. Importar Dataset


In [1]:
import pandas as pd

# Cargar los datos
# Asegúrate de que los archivos 'train.csv' y 'test.csv' estén en la misma carpeta que tu código
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

train_df.head()

,ID,RevolvingUtilizationOfUnsecuredLines,Age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,SeriousDlqin2yrs
0,9580,0.668999,58,2,0.449504,3425.0,9,1,1,1,1.0,0
1,39755,0.015922,71,0,6.000000,NaN,5,0,0,0,0.0,0
2,118799,0.183062,52,1,0.035593,5000.0,9,0,0,0,0.0,0
3,16489,0.162301,77,0,0.227886,2000.0,8,0,0,0,0.0,0
4,149857,0.404199,30,0,0.026010,5843.0,4,0,0,0,0.0,0


In [2]:
# Un vistazo rápido a los valores nulos en el conjunto de entrenamiento
print("Valores nulos en Train:")
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])

Valores nulos en Train:
MonthlyIncome         20836
NumberOfDependents     2764
dtype: int64


### 2. Limpieza

In [3]:
# Rellenar nulos en MonthlyIncome con la mediana
train_df['MonthlyIncome'] = train_df['MonthlyIncome'].fillna(train_df['MonthlyIncome'].median())
test_df['MonthlyIncome'] = test_df['MonthlyIncome'].fillna(train_df['MonthlyIncome'].median()) # Usamos la mediana de train para evitar fugas de datos (data leakage)

# Rellenar nulos en NumberOfDependents con 0
train_df['NumberOfDependents'] = train_df['NumberOfDependents'].fillna(0)
test_df['NumberOfDependents'] = test_df['NumberOfDependents'].fillna(0)

# Separar las características (X) de la variable a predecir (y)
# Quitamos el 'ID' porque no aporta información predictiva, es solo un identificador
X = train_df.drop(columns=['ID', 'SeriousDlqin2yrs'])
y = train_df['SeriousDlqin2yrs']

# Preparamos también el set de test final
X_test = test_df.drop(columns=['ID'])

### 3. Modelos

In [4]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import roc_auc_score
import time

# Dividimos en Train y Validation para tener una prueba final antes de Kaggle
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 1. Definir los modelos base
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
xgb = XGBClassifier(random_state=42, eval_metric='auc', n_jobs=-1)
lgbm = LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)

# 2. Definir los espacios de búsqueda (Grids)
# Mantenemos las opciones acotadas para no tardar una eternidad
rf_params = {
    'n_estimators': [100, 250],
    'max_depth': [5, 8]
}

xgb_params = {
    'n_estimators': [100, 250],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1]
}

lgbm_params = {
    'n_estimators': [100, 250],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1]
}

# 3. Diccionario para iterar
model_grids = {
    'RandomForest': (rf, rf_params),
    'XGBoost': (xgb, xgb_params),
    'LightGBM': (lgbm, lgbm_params)
}

best_models = {}
print("🏁 ¡Arranca la carrera hacia la precisión! 🏁\n")

# 4. Bucle mágico de Grid Search
for name, (model, params) in model_grids.items():
    print(f"Entrenando y optimizando {name}...")
    start_time = time.time()
    
    # Configuramos el GridSearch para maximizar el 'roc_auc'
    # cv=3 significa Cross Validation de 3 pliegues (para ahorrar tiempo)
    grid = GridSearchCV(estimator=model, param_grid=params, scoring='roc_auc', cv=3, n_jobs=-1)
    grid.fit(X_train, y_train)
    
    # Guardamos el mejor modelo
    best_models[name] = grid.best_estimator_
    
    # Calculamos el tiempo
    elapsed_time = time.time() - start_time
    
    print(f"✅ {name} completado en {elapsed_time:.2f} segundos.")
    print(f"Mejores parámetros: {grid.best_params_}")
    print(f"Mejor AUC en Cross-Validation: {grid.best_score_:.4f}\n")
    print("-" * 50)

🏁 ¡Arranca la carrera hacia la precisión! 🏁

Entrenando y optimizando RandomForest...
✅ RandomForest completado en 17.13 segundos.
Mejores parámetros: {'max_depth': 8, 'n_estimators': 100}
Mejor AUC en Cross-Validation: 0.8610

--------------------------------------------------
Entrenando y optimizando XGBoost...
✅ XGBoost completado en 4.96 segundos.
Mejores parámetros: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 250}
Mejor AUC en Cross-Validation: 0.8628

--------------------------------------------------
Entrenando y optimizando LightGBM...
✅ LightGBM completado en 5.89 segundos.
Mejores parámetros: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 250}
Mejor AUC en Cross-Validation: 0.8627

--------------------------------------------------


### 4. Evaluación

In [5]:
# 1. Evaluar todos en el set de validación
print("📊 Evaluando en el set de Validación:")
best_val_auc = 0
champion_name = ""
champion_model = None

for name, model in best_models.items():
    # Predecimos probabilidades (necesitamos la probabilidad de la clase 1)
    preds = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, preds)
    print(f"{name} AUC Validación: {auc:.4f}")
    
    # Buscamos al ganador
    if auc > best_val_auc:
        best_val_auc = auc
        champion_name = name
        champion_model = model

print(f"\n🏆 ¡El ganador indiscutible es {champion_name} con un AUC de {best_val_auc:.4f}! 🏆")

# 2. Generar el archivo de sumisión con el campeón
print(f"\nGenerando predicciones para Kaggle usando {champion_name}...")
# Re-entrenamos el campeón con TODOS los datos (X, y) para aprovechar toda la info
champion_model.fit(X, y)

# Predecimos sobre el test real
test_preds = champion_model.predict_proba(X_test)[:, 1]

# 3. Creamos el DataFrame y lo guardamos
submission = pd.DataFrame({
    'ID': test_df['ID'],
    'SeriousDlqin2yrs': test_preds
})

submission.to_csv('submission_campeon.csv', index=False)
print("¡Archivo 'submission_campeon.csv' listo para subir a Kaggle! Enséñales la pasta. 💸")

📊 Evaluando en el set de Validación:
RandomForest AUC Validación: 0.8583
XGBoost AUC Validación: 0.8599
LightGBM AUC Validación: 0.8601

🏆 ¡El ganador indiscutible es LightGBM con un AUC de 0.8601! 🏆

Generando predicciones para Kaggle usando LightGBM...
¡Archivo 'submission_campeon.csv' listo para subir a Kaggle! Enséñales la pasta. 💸
